# 13 · Safari species hotspots: buffalo, elephant, lion, cheetah, leopard

The buffalo recipe from 08–10, applied to all five target species from notebook 02:

- **Data:** GBIF sightings inside the Serengeti ecosystem's protected areas, 2017–2025, thinned to one per 1 km cell per year.
- **Background:** random points in the visited 10 km cells inside the same protected areas, year-matched, 10 per presence.
- **Model:** regularized XGBoost on AlphaEarth embeddings + habitat covariates (without elevation), 10's final model.
- **Checks:** 10 km block CV, unseen regions (k-means), the **coordinates-only floor**, and the **camera traps** with the camera region held out.
- **Map:** 3 compact hotspots per species, with that species' camera photo rates as context.

To save Earth Engine work, all species share **one background pool**. Each species then draws its own year-matched background from it, leaving out pool points within 700 m of its same-year sightings. Buffalo is re-run this way too, so all five are comparable; its numbers differ slightly from 10's.

## 0. Data

Everything that already exists comes from the cache: the GBIF mammal records with species names (11), the camera traps with embeddings and covariates (09), and the 2.5 km map grid (09). Only the new species' sightings and the shared background pool need Earth Engine, once.

In [1]:
import ee
import numpy as np
import pandas as pd

import buffalo_data as bd
import buffalo_validation as bv
from buffalo_data import BAND_COLS, COVARIATE_COLS, BuffaloConfig, build_buffalo_dataset

EE_PROJECT = "project-0cf410a3-f35f-4911-9c5"
ee.Initialize(project=EE_PROJECT)

config = BuffaloConfig(cache_dir="cache")
data = build_buffalo_dataset(config)
areas = bv.fetch_protected_areas()
records = bd.fetch_mammal_records(config)

MAP_YEAR, GRID_SPACING_M = 2025, 2_500
grid = bd.cached(config, f"serengeti_grid_{MAP_YEAR}_{GRID_SPACING_M}",
                 lambda: bv.serengeti_map_grid(data.visited_cells, areas, config, MAP_YEAR, GRID_SPACING_M))
grid = grid.dropna(subset=BAND_COLS + COVARIATE_COLS).reset_index(drop=True)
cam = bv.add_utm(data.cameras, config.cell_size_m)
cam = bd.cached(config, "camera_covariates", lambda: bd.add_covariates(cam, "SiteID", config))

SPECIES = bv.SAFARI_SPECIES
presences = {name: bv.serengeti_presences(records, sp["speciesKey"], areas, config) for name, sp in SPECIES.items()}
print(pd.DataFrame({name: {"Serengeti sightings (thinned)": len(p)} for name, p in presences.items()}).T)

presences: loaded from cache\buffalo_presences_8be5ffc111.parquet
buffalo sightings in box: 674
presences after thinning: 658
mammals: loaded from cache\buffalo_mammals_8be5ffc111.parquet
mammal sightings in box: 15873
visited cells: 334


points_embedded: loaded from cache\buffalo_points_embedded_8be5ffc111.parquet
points missing embeddings: 0
cameras_embedded: loaded from cache\buffalo_cameras_embedded_8be5ffc111.parquet
dataset: 658 presences, 6580 background, 222 cameras


mammal_records: loaded from cache\buffalo_mammal_records_8be5ffc111.parquet
mammal records in box: 15873
serengeti_grid_2025_2500: loaded from cache\buffalo_serengeti_grid_2025_2500_8be5ffc111.parquet
camera_covariates: loaded from cache\buffalo_camera_covariates_8be5ffc111.parquet
          Serengeti sightings (thinned)
buffalo                             266
elephant                            475
lion                               1042
cheetah                             457
leopard                             365


### Shared background pool

Per year, the pool holds 12 × the largest number of sightings any species has that year: 10 per presence, plus 20 % spare for the points each species leaves out near its own sightings. The pool and all species' sightings then get embeddings and covariates with the same recipes as before, in chunks of 1,000 points per Earth Engine request.

In [2]:
GEE_CHUNK = 1_000
counts = pd.DataFrame({name: p.groupby("year").size() for name, p in presences.items()}).fillna(0)
pool_per_year = (counts.max(axis=1) * 12).astype(int).to_dict()


def build_points():
    pool = bv.serengeti_background_pool(pool_per_year, data.visited_cells, areas, config)
    sightings = pd.concat([p.rename(columns={"gbifID": "point_id"}).assign(species=name) for name, p in presences.items()],
                          ignore_index=True)
    points = pd.concat([sightings, pool.assign(species="pool")], ignore_index=True)
    print("points to embed:", len(points))
    points = pd.concat([bd.embed_points(points.iloc[s:s + GEE_CHUNK], "point_id", config) for s in range(0, len(points), GEE_CHUNK)],
                       ignore_index=True)
    return bd.add_covariates(points, "point_id", config)


points = bd.cached(config, "safari_points", build_points).dropna(subset=BAND_COLS + COVARIATE_COLS)
pool = points[points["species"] == "pool"].drop(columns="species")
print(f"pool: {len(pool)} background points; sightings with features:", points[points["species"] != "pool"]["species"].value_counts().to_dict())

points to embed: 15109


2017: 143 points embedded


2018: 187 points embedded


2019: 135 points embedded


2020: 33 points embedded


2021: 34 points embedded


... (67 outputs left out)

C:\Users\Juan\scoop\apps\python313\current\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


C:\Users\Juan\scoop\apps\python313\current\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


2025: 2880 points, covariates
pool: 12504 background points; sightings with features: {'lion': 1042, 'elephant': 475, 'cheetah': 457, 'leopard': 365, 'buffalo': 266}


C:\Users\Juan\scoop\apps\python313\current\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


## 1. Model checks per species

For every species, the same checks as in 08–10:
- **block AUC:** 10 km blocks held out (gaps inside known regions)
- **region AUC:** 4 k-means regions of that species' sightings held out (unseen regions)
- **floor region AUC:** the same, with only the x/y coordinates. A habitat model has to beat this to show it learned more than "central vs remote".
- **camera rho:** each camera ranked by the model that never saw its region, against that species' camera photo rate. `floor camera rho` is the same for the coordinates-only model.

The camera traps are independent, effort-corrected data, so **camera rho above the floor** is the main evidence that a species' map means something. For rare camera species (cheetah, leopard), most cameras never photographed the species, so the camera check is weaker there.

In [3]:
MODEL_FEATURES = BAND_COLS + [c for c in COVARIATE_COLS if c != "elevation_m"]
SETTING = bv.XGB_SETTINGS["regularized"]
K_REGIONS = 4

datasets, camera_tables, checks = {}, {}, {}
for name, sp in SPECIES.items():
    sightings = points[points["species"] == name].drop(columns="species")
    df, ratio = bv.species_dataset(sightings, pool, config)
    rates = bd.camera_species_rate(sp["camera_labels"], config)
    cam_sp = cam.assign(rate=cam["SiteID"].map(rates)).dropna(subset=["rate"])
    habitat, _ = bv.region_evaluation(df, cam_sp, K_REGIONS, SETTING, MODEL_FEATURES)
    floor, _ = bv.region_evaluation(df, cam_sp, K_REGIONS, SETTING, ["x", "y"])
    datasets[name], camera_tables[name] = df, cam_sp
    checks[name] = {"presences": int(df["presence"].sum()), "background per presence": ratio,
                    "cameras with the species": int((cam_sp["rate"] > 0).sum()),
                    "block AUC": habitat["block AUC"], "region AUC": habitat["region AUC"], "floor region AUC": floor["region AUC"],
                    "camera rho": habitat["camera rho"], "floor camera rho": floor["camera rho"]}
    print(f"{name}: done")

check_table = pd.DataFrame(checks).T
check_table["beats floor on cameras"] = check_table["camera rho"] > check_table["floor camera rho"]
check_table.round(3)

buffalo: done


elephant: done


lion: done


cheetah: done


leopard: done


,presences,background per presence,cameras with the species,block AUC,region AUC,floor region AUC,camera rho,floor camera rho,beats floor on cameras
buffalo,266.0,10.0,182.0,0.731,0.636,0.629,0.391,0.441,False
elephant,475.0,10.0,187.0,0.720,0.613,0.615,0.202,0.342,False
lion,1042.0,10.0,171.0,0.750,0.629,0.649,-0.123,0.033,False
cheetah,457.0,10.0,109.0,0.697,0.577,0.596,-0.166,0.014,False
leopard,365.0,10.0,67.0,0.754,0.628,0.748,0.207,0.029,True


**Reading it:**
- **camera rho clearly above 0 and above the floor:** the map ranks places in a way that agrees with independent camera data. That's the strongest result possible here.
- **region AUC near or below the floor:** in unseen regions the model does no better than "central vs remote". Its hotspots mostly say where the species gets *reported*.
- **Few cameras with the species** (cheetah, leopard): the camera rho rests on few non-zero cameras. Treat it as weak evidence either way.

## 2. Hotspots per species

Each species' model is fit on all its Serengeti data and scores the 2.5 km map grid for `MAP_YEAR`. The 3 hotspots come from the same picking as 10 (`bv.hotspot_areas`): smoothed over 3 km, top 10 %, at most 4 km from each peak. The cameras inside each hotspot show that species' photo rate as context, compared with its median over all cameras.

In [4]:
N_AREAS, HOT_PERCENT, PEAK_RADIUS_M, SMOOTH_SIGMA_M = 3, 10, 4_000, 3_000

hotspots = {}
for name, df in datasets.items():
    model = bv.make_model(df["presence"], SETTING).fit(df[MODEL_FEATURES], df["presence"])
    score = model.predict_proba(grid[MODEL_FEATURES])[:, 1]
    hotspots[name] = bv.hotspot_areas(grid, score, camera_tables[name], GRID_SPACING_M, SMOOTH_SIGMA_M, HOT_PERCENT,
                                      PEAK_RADIUS_M, N_AREAS)
    print(f"--- {name} (median camera rate over all cameras: {camera_tables[name]['rate'].median():.2f} per 100 trap-days)")
    print(hotspots[name].drop(columns="geometry").round(3).to_string(index=False))

--- buffalo (median camera rate over all cameras: 4.55 per 100 trap-days)
 rank  area_km2  percentile  peak_lon  peak_lat  cameras  camera median rate
    1     56.25      98.589    34.855    -2.498       12               8.979
    2     56.25      97.425    34.945    -2.543       14               4.496
    3     43.75      96.074    34.877    -2.543        7               0.000


--- elephant (median camera rate over all cameras: 5.18 per 100 trap-days)
 rank  area_km2  percentile  peak_lon  peak_lat  cameras  camera median rate
    1     56.25      98.292    34.877    -2.475       12               6.988
    2     56.25      99.274    34.967    -2.520       13               5.525
    3     37.50      96.788    34.877    -2.430        6               9.581


--- lion (median camera rate over all cameras: 0.57 per 100 trap-days)
 rank  area_km2  percentile  peak_lon  peak_lat  cameras  camera median rate
    1     56.25      99.505    34.967    -2.543       12               6.016
    2     56.25      98.696    34.877    -2.543        9               1.325
    3     43.75      98.127    34.855    -2.498       10               0.764


--- cheetah (median camera rate over all cameras: 0.00 per 100 trap-days)
 rank  area_km2  percentile  peak_lon  peak_lat  cameras  camera median rate
    1     56.25      99.691    34.967    -2.520       13               1.328
    2     37.50      98.341    34.967    -2.566        6               1.880
    3     37.50      95.971    34.967    -2.475       10               0.000


--- leopard (median camera rate over all cameras: 0.00 per 100 trap-days)
 rank  area_km2  percentile  peak_lon  peak_lat  cameras  camera median rate
    1     56.25      98.667    34.855    -2.566       10                 0.0
    2     56.25      98.218    34.967    -2.566       11                 0.0
    3     37.50      99.127    34.967    -2.520        8                 0.0


## 3. Map

One species at a time: pick it in the layer switch (top right). Each shows its 3 hotspots, numbered 1 (darkest) to 3, and the camera traps sized by **that species'** photo rate. Hover a hotspot for its size, score and the cameras inside; hover a camera for its rate. Click anywhere for lat/lon.

In [5]:
import folium

light, dark = np.array([0x86, 0xb6, 0xef]), np.array([0x0d, 0x36, 0x6b])
colour = lambda rank: "#%02x%02x%02x" % tuple((dark + (light - dark) * (rank - 1) / max(N_AREAS - 1, 1)).astype(int))

species_map = folium.Map(location=[grid["decimalLatitude"].mean(), grid["decimalLongitude"].mean()], zoom_start=8, tiles=None)
folium.TileLayer("OpenStreetMap", control=False).add_to(species_map)

for name in SPECIES:
    layer = folium.FeatureGroup(name=name, overlay=False).add_to(species_map)
    for _, a in hotspots[name].iterrows():
        c = colour(a["rank"])
        note = (f"{a['cameras']} cameras inside, median {a['camera median rate']:.1f} photos per 100 trap-days"
                if a["cameras"] else "no cameras inside")
        folium.GeoJson(a.geometry.__geo_interface__,
                       style_function=lambda _, c=c: {"fillColor": c, "color": c, "weight": 1.5, "fillOpacity": 0.55},
                       tooltip=f"{name} hotspot {a['rank']}: {a['area_km2']:.0f} km², percentile {a['percentile']:.0f}; {note}").add_to(layer)
        folium.Marker([a["peak_lat"], a["peak_lon"]], icon=folium.DivIcon(
            html=f'<div style="font:bold 13px sans-serif;color:#fff;background:{c};border-radius:50%;width:22px;height:22px;'
                 f'line-height:22px;text-align:center;border:1.5px solid #fff">{a["rank"]}</div>',
            icon_size=(22, 22), icon_anchor=(11, 11))).add_to(layer)
    cams = camera_tables[name]
    cap = max(cams["rate"].quantile(0.95), 1e-9)
    for _, cm in cams.iterrows():
        folium.CircleMarker([cm["decimalLatitude"], cm["decimalLongitude"]], radius=2 + 6 * min(cm["rate"], cap) / cap,
                            color="#1f1f1f", weight=1, fill=True, fill_color="#1f1f1f", fill_opacity=0.6,
                            tooltip=f"camera {cm['SiteID']}: {cm['rate']:.2f} {name} photos per 100 trap-days").add_to(layer)

folium.LayerControl(collapsed=False).add_to(species_map)
folium.LatLngPopup().add_to(species_map)
species_map

**Reading the map, honestly:**
- **A hotspot whose cameras photograph the species above its median** is supported by independent data. A hotspot whose cameras rarely see the species is contradicted; treat it as doubtful.
- **Hotspots without cameras** rest on the model alone. Check the species' camera rho and floor in section 1 before trusting them.
- **All maps cover only the visited Serengeti** (2025 landscape). They rank places against each other ("more likely / less likely"); they're not probabilities, and they say nothing about unvisited areas.